CELL 1 — Install

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn -q
print('Done')

Done


CELL 2 — Imports

In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
import re
import warnings
from collections import Counter
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import seaborn as sns

print('All imports done')

All imports done


CELL 3 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


CELL 4 — Set Paths

In [ ]:
D1 = '/content/drive/MyDrive/resume_checker_data/Dataset 1 (LinkedIn Job Postings)/'
D2 = '/content/drive/MyDrive/resume_checker_data/Dataset 2 - Campus Placement/'
OUT = '/content/drive/MyDrive/resume_checker_data/models/'

import os
os.makedirs(OUT, exist_ok=True)
print('Paths set')

Paths set


CELL 5 — Load Files

In [ ]:
print('Loading postings.csv')
postings = pd.read_csv(
    D1 + 'postings.csv',
    usecols=['job_id', 'title', 'description', 'normalized_salary', 'min_salary', 'max_salary']
)
print(f'postings  : {len(postings):,}')

job_skills = pd.read_csv(D1 + 'job_skills.csv')
skills     = pd.read_csv(D1 + 'skills.csv')
print(f'job_skills: {len(job_skills):,}')
print(f'skills    : {len(skills)}')

D2 = '/content/drive/MyDrive/resume_checker_data/Dataset 2 — Campus Placement/'
placement = pd.read_csv(D2 + 'Placement_Data_Full_Class.csv')
print(f'placement : {len(placement)}')

Loading postings.csv
postings  : 123,849
job_skills: 213,768
skills    : 35
placement : 215


CELL 6 — Label Roles

In [ ]:
ROLE_KEYWORDS = {
    'Software Engineer':         ['software engineer', 'software developer', 'backend engineer', 'frontend engineer'],
    'Full Stack Developer':      ['full stack', 'fullstack', 'full-stack'],
    'Frontend Developer':        ['frontend developer', 'front end developer', 'react developer', 'ui developer'],
    'Backend Developer':         ['backend developer', 'back end developer', 'node developer'],
    'Data Scientist':            ['data scientist', 'data science'],
    'Data Analyst':              ['data analyst', 'business analyst', 'analytics analyst'],
    'Machine Learning Engineer': ['machine learning', 'ml engineer', 'ai engineer', 'deep learning'],
    'Data Engineer':             ['data engineer', 'etl developer', 'big data'],
    'DevOps Engineer':           ['devops', 'platform engineer', 'site reliability', 'sre'],
    'Cloud Engineer':            ['cloud engineer', 'aws engineer', 'azure engineer', 'gcp engineer'],
    'Cybersecurity Analyst':     ['security analyst', 'cybersecurity', 'information security', 'penetration tester'],
    'Mobile Developer':          ['mobile developer', 'android developer', 'ios developer', 'flutter developer'],
    'QA Engineer':               ['qa engineer', 'quality assurance', 'test engineer', 'automation engineer'],
    'Product Manager':           ['product manager', 'product owner'],
    'Project Manager':           ['project manager', 'scrum master'],
    'UI/UX Designer':            ['ui/ux', 'ux designer', 'ui designer', 'product designer'],
    'Database Administrator':    ['database administrator', 'dba', 'sql developer'],
    'Network Engineer':          ['network engineer', 'network administrator'],
    'Marketing Manager':         ['marketing manager', 'digital marketing', 'growth marketer'],
}

def map_role(title):
    t = str(title).lower()
    for role, kws in ROLE_KEYWORDS.items():
        if any(k in t for k in kws):
            return role
    return None

postings['role'] = postings['title'].apply(map_role)
labeled = postings.dropna(subset=['role', 'description'])

print(f'Total labeled samples: {len(labeled):,}')
print(labeled['role'].value_counts().to_string())

Total labeled samples: 9,595
role
Project Manager              2084
Software Engineer            1624
Data Analyst                  964
QA Engineer                   687
Product Manager               658
Marketing Manager             486
Data Engineer                 433
DevOps Engineer               402
Data Scientist                349
Cybersecurity Analyst         325
Full Stack Developer          319
Network Engineer              292
Database Administrator        243
Machine Learning Engineer     174
UI/UX Designer                143
Frontend Developer            136
Cloud Engineer                116
Backend Developer              87
Mobile Developer               73


CELL 7 — Build Features

In [ ]:
labeled = labeled.merge(
    job_skills.merge(skills, on='skill_abr', how='left')
    .groupby('job_id')['skill_name']
    .apply(lambda x: ' '.join(x.dropna()))
    .reset_index()
    .rename(columns={'skill_name': 'skills_text'}),
    on='job_id', how='left'
)
labeled['skills_text'] = labeled['skills_text'].fillna('')

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

labeled['combined_text'] = (
    labeled['description'].apply(clean_text) + ' ' +
    labeled['skills_text'].apply(clean_text) + ' ' +
    labeled['skills_text'].apply(clean_text) + ' ' +
    labeled['skills_text'].apply(clean_text)
)

print(f'Features ready: {len(labeled):,} samples')
print(f'Sample: {labeled["combined_text"].iloc[0][:200]}')

Features ready: 9,595 samples
Sample: a leading pharmaceutical company committed to developing and commercializing innovative and high quality medicines that improve the lives of patients is now hiring for a senior product marketing manag


CELL 8 — TF-IDF Vectorize

In [ ]:
X_text  = labeled['combined_text'].values
y_roles = labeled['role'].values

role_encoder = LabelEncoder()
y_encoded    = role_encoder.fit_transform(y_roles)

tfidf = TfidfVectorizer(
    max_features=8000,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_tfidf = tfidf.fit_transform(X_text)
print(f'TF-IDF matrix: {X_tfidf.shape}')
print(f'Roles: {list(role_encoder.classes_)}')

TF-IDF matrix: (9595, 8000)
Roles: ['Backend Developer', 'Cloud Engineer', 'Cybersecurity Analyst', 'Data Analyst', 'Data Engineer', 'Data Scientist', 'Database Administrator', 'DevOps Engineer', 'Frontend Developer', 'Full Stack Developer', 'Machine Learning Engineer', 'Marketing Manager', 'Mobile Developer', 'Network Engineer', 'Product Manager', 'Project Manager', 'QA Engineer', 'Software Engineer', 'UI/UX Designer']


CELL 9 — Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)
print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

Train: 7,676 | Test: 1,919


CELL 10 — Train Random Forest

In [ ]:
print('Training Random Forest... (2-3 mins)')

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
acc    = accuracy_score(y_test, y_pred)

print(f'Done!')
print(f'Accuracy: {acc*100:.2f}%')
print(classification_report(y_test, y_pred, target_names=role_encoder.classes_))

Training Random Forest... (2-3 mins)
Done!
Accuracy: 87.91%
                           precision    recall  f1-score   support

        Backend Developer       1.00      0.29      0.45        17
           Cloud Engineer       0.82      0.39      0.53        23
    Cybersecurity Analyst       0.97      0.91      0.94        65
             Data Analyst       0.88      0.95      0.92       193
            Data Engineer       0.88      0.91      0.89        86
           Data Scientist       0.96      0.79      0.87        70
   Database Administrator       0.97      0.78      0.86        49
          DevOps Engineer       0.80      0.75      0.77        80
       Frontend Developer       0.94      0.59      0.73        27
     Full Stack Developer       0.76      0.44      0.55        64
Machine Learning Engineer       0.79      0.74      0.76        35
        Marketing Manager       0.98      0.96      0.97        97
         Mobile Developer       1.00      0.53      0.70        15
 

CELL 11 — Cross Validation

In [ ]:
cv = cross_val_score(rf, X_tfidf, y_encoded, cv=5, scoring='accuracy', n_jobs=-1)
print(f'5-Fold CV: {cv.mean()*100:.2f}% ± {cv.std()*100:.2f}%')
print(f'Folds: {[f"{s*100:.1f}%" for s in cv]}')

5-Fold CV: 87.31% ± 1.03%
Folds: ['86.1%', '86.5%', '88.9%', '88.1%', '86.9%']


CELL 12 — Build Cosine Centroids

In [ ]:
role_centroids = {}
role_profiles  = {}

for role in role_encoder.classes_:
    mask        = labeled['role'] == role
    role_matrix = tfidf.transform(labeled[mask]['combined_text'].values)
    centroid    = np.asarray(role_matrix.mean(axis=0))
    role_centroids[role] = centroid

    salary_data = labeled[mask]['normalized_salary'].dropna()
    skill_words = ' '.join(labeled[mask]['skills_text'].dropna().tolist()).split()
    top_skills  = [s for s, _ in Counter(skill_words).most_common(8) if len(s) > 2]

    role_profiles[role] = {
        'required_skills': top_skills,
        'avg_salary': round(salary_data.mean(), 0) if len(salary_data) > 0 else None,
        'min_salary': round(salary_data.quantile(0.25), 0) if len(salary_data) > 0 else None,
        'max_salary': round(salary_data.quantile(0.75), 0) if len(salary_data) > 0 else None,
        'sample_count': int(mask.sum())
    }

print(f'Centroids built for {len(role_centroids)} roles')

Centroids built for 19 roles


CELL 13 — Test Both Layers

In [ ]:
def predict_resume(resume_text, top_n=5):
    cleaned    = clean_text(resume_text)
    resume_vec = tfidf.transform([cleaned])
    resume_arr = np.asarray(resume_vec.todense())

    rf_idx        = rf.predict(resume_vec)[0]
    rf_role       = role_encoder.inverse_transform([rf_idx])[0]
    rf_confidence = round(rf.predict_proba(resume_vec)[0][rf_idx] * 100, 2)

    scores = {role: round(float(cosine_similarity(resume_arr, c)[0][0]) * 100, 2)
              for role, c in role_centroids.items()}
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    print(f'RF Prediction : {rf_role} ({rf_confidence}% confident)')
    print(f'Cosine Ranking:')
    for i, (role, score) in enumerate(ranked[:top_n], 1):
        tag = ' ← RF agrees' if role == rf_role else ''
        print(f'   {i}. {role:<35} {score:.2f}%{tag}')

predict_resume("Python TensorFlow deep learning NLP machine learning neural networks scikit-learn pandas")
print()
predict_resume("React JavaScript HTML CSS TypeScript frontend UI components Next.js Tailwind")

RF Prediction : Machine Learning Engineer (60.0% confident)
Cosine Ranking:
   1. Machine Learning Engineer           32.27% ← RF agrees
   2. Data Scientist                      17.09%
   3. Data Engineer                       4.89%
   4. Software Engineer                   4.40%
   5. Network Engineer                    3.70%

RF Prediction : Frontend Developer (47.0% confident)
Cosine Ranking:
   1. Frontend Developer                  32.13% ← RF agrees
   2. Full Stack Developer                17.50%
   3. Backend Developer                   10.66%
   4. Software Engineer                   8.76%
   5. UI/UX Designer                      7.85%



CELL 14 — Save Part 1 Models

In [ ]:
with open(OUT + 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open(OUT + 'job_role_classifier.pkl', 'wb') as f:
    pickle.dump({'classifier': rf, 'role_encoder': role_encoder,
                 'roles': list(role_encoder.classes_)}, f)

with open(OUT + 'cosine_centroids.pkl', 'wb') as f:
    pickle.dump({'centroids': role_centroids, 'roles': list(role_centroids.keys())}, f)

with open(OUT + 'job_roles.json', 'w') as f:
    json.dump(role_profiles, f, indent=2, default=str)

print('tfidf_vectorizer.pkl saved')
print('job_role_classifier.pkl saved')
print('cosine_centroids.pkl saved')
print('job_roles.json saved')

tfidf_vectorizer.pkl saved
job_role_classifier.pkl saved
cosine_centroids.pkl saved
job_roles.json saved


CELL 15 — Preprocess Placement Data

In [ ]:
FEATURES = ['ssc_p', 'hsc_p', 'degree_p', 'etest_p', 'mba_p',
            'gender', 'ssc_b', 'hsc_b', 'hsc_s', 'degree_t', 'workex', 'specialisation']
CAT_COLS  = ['gender', 'ssc_b', 'hsc_b', 'hsc_s', 'degree_t', 'workex', 'specialisation']

X_p = placement[FEATURES].copy()
y_p = (placement['status'] == 'Placed').astype(int)

placement_encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    X_p[col] = le.fit_transform(X_p[col].astype(str))
    placement_encoders[col] = le

print(f'Ready: {X_p.shape} | Placed: {y_p.sum()} | Not Placed: {(y_p==0).sum()}')

Ready: (215, 12) | Placed: 148 | Not Placed: 67


CELL 16 — Train Placement Model

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_p, y_p, test_size=0.2, random_state=42, stratify=y_p)

placement_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
placement_pipeline.fit(X_tr, y_tr)

y_pr  = placement_pipeline.predict(X_te)
acc_p = accuracy_score(y_te, y_pr)
auc_p = roc_auc_score(y_te, placement_pipeline.predict_proba(X_te)[:, 1])
cv_p  = cross_val_score(placement_pipeline, X_p, y_p, cv=5, scoring='accuracy')

print(f'   Accuracy : {acc_p*100:.2f}%')
print(f'   ROC-AUC  : {auc_p:.4f}')
print(f'   5-Fold CV: {cv_p.mean()*100:.2f}% ± {cv_p.std()*100:.2f}%')
print(classification_report(y_te, y_pr, target_names=['Not Placed', 'Placed']))

   Accuracy : 83.72%
   ROC-AUC  : 0.9333
   5-Fold CV: 84.65% ± 4.31%
              precision    recall  f1-score   support

  Not Placed       0.67      0.92      0.77        13
      Placed       0.96      0.80      0.87        30

    accuracy                           0.84        43
   macro avg       0.81      0.86      0.82        43
weighted avg       0.87      0.84      0.84        43



CELL 17 — Save Placement Model

In [ ]:
with open(OUT + 'placement_model.pkl', 'wb') as f:
    pickle.dump({
        'pipeline':       placement_pipeline,
        'label_encoders': placement_encoders,
        'features':       FEATURES,
        'cat_cols':       CAT_COLS,
        'accuracy':       round(acc_p * 100, 2),
        'categories':     {col: list(le.classes_) for col, le in placement_encoders.items()}
    }, f)

print('placement_model.pkl saved')
print(f'\n🎉 All models saved to: {OUT}')

placement_model.pkl saved

🎉 All models saved to: /content/drive/MyDrive/resume_checker_data/models/
